In [ ]:
!nvidia-smi

Sun Jun 28 02:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

I. Mount & Copy file vào Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/Thesis/rice_disease_project.zip" -d "/content/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/1539_rice_disease_project/data/yolo_dataset/images/train/healthy_00897_healthy-366_JPG.rf.JsAgTMKvkFadohq1vOzg.JPG  
  inflating: /content/__MACOSX/1539_rice_disease_project/data/yolo_dataset/images/train/._healthy_00897_healthy-366_JPG.rf.JsAgTMKvkFadohq1vOzg.JPG  
  inflating: /content/1539_rice_disease_project/data/yolo_dataset/images/train/healthy_00744_healthy-108_JPG.rf.dimXoobFMczjOZUC2W5q.JPG  
  inflating: /content/__MACOSX/1539_rice_disease_project/data/yolo_dataset/images/train/._healthy_00744_healthy-108_JPG.rf.dimXoobFMczjOZUC2W5q.JPG  
  inflating: /content/1539_rice_disease_project/data/yolo_dataset/images/train/leaf_scald_01608_leaf_scald (55)_pp_jpg.rf.76x4XqJxMADp6LsUkGoJ.jpg  
  inflating: /content/__MACOSX/1539_rice_disease_project/data/yolo_dataset/images/train/._leaf_scald_01608_leaf_scald (55)_pp_jpg.rf.76x4XqJxMADp6LsUkGoJ.jpg  
  inflating: /content/1539_rice_disease_project/data/yolo_data

II. Setup môi trường ở Colab

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.4 MB/s eta 0:00:00


In [ ]:
!pip install -r /content/rice_disease_project/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00


III. Merge dataset trong folder /raw_datasets

In [ ]:
!python /content/rice_disease_project/src/utils/merge_datasets.py

STARTING DATASET MERGE
Processing: bacterial_leaf_blight
  train: +320 images
  val: +40 images
  test: +40 images
Processing: brown_spot
  train: +340 images
  val: +43 images
  test: +42 images
Processing: healthy
  train: +320 images
  val: +40 images
  test: +40 images
Processing: leaf_blast
  train: +336 images
  val: +42 images
  test: +42 images
Processing: leaf_scald
  train: +320 images
  val: +40 images
  test: +40 images
Processing: sheath_blight
  train: +175 images
  val: +22 images
  test: +22 images
DATASET SUMMARY
bacterial_leaf_blight          train= 320 val=  40 test=  40 total= 400
brown_spot                     train= 340 val=  43 test=  42 total= 425
healthy                        train= 320 val=  40 test=  40 total= 400
leaf_blast                     train= 336 val=  42 test=  42 total= 420
leaf_scald                     train= 320 val=  40 test=  40 total= 400
sheath_blight                  train= 175 val=  22 test=  22 total= 219
TOTAL                          t

IV. Train YOLO V8

Xoá thư mục

In [ ]:
import shutil
import os
folder_path = ''
# Kiểm tra xem thư mục có tồn tại không trước khi xóa
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print(f"Đã xoá thành công toàn bộ thư mục: {folder_path}")
else:
    print(f"Thư mục không tồn tại: {folder_path}")

Đã xoá thành công toàn bộ thư mục: /content/rice_disease_project/data/yolo_dataset


- Option

1. GPU


* Lấy model từ file config

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/train_yolo.py --model yolo9c

TRAIN YOLO - model = yolo9c
Pretrained: yolov9c.pt
Data: /content/rice_disease_project/data/yolo_dataset/data.yaml
Epochs: 100
Image size: 640
Batch size: 16
Device: 0
Run name: rice_yolo9c
Best weights -> /content/rice_disease_project/models/yolo/yolo9c.pt
Classes: 6 active / 6 total
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rice_disease_project/data/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/train_yolo.py --model yolo9e

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
TRAIN YOLO - model = yolo9e
Pretrained: yolov9e.pt
Data: /content/rice_disease_project/data/yolo_dataset/data.yaml
Epochs: 100
Image size: 640
Batch size: 16
Device: 0
Run name: rice_yolo9e
Best weights -> /content/rice_disease_project/models/yolo/yolo9e.pt
Classes: 6 active / 6 total
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ri

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/train_yolo.py --model yolo10l

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
TRAIN YOLO - model = yolo10l
Pretrained: yolov10l.pt
Data: /content/rice_disease_project/data/yolo_dataset/data.yaml
Epochs: 100
Image size: 640
Batch size: 16
Device: 0
Run name: rice_yolo10l
Best weights -> /content/rice_disease_project/models/yolo/yolo10l.pt
Classes: 6 active / 6 total
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/conten

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/train_yolo.py --model yolo10x

TRAIN YOLO - model = yolo10x
Pretrained: yolov10x.pt
Data: /content/rice_disease_project/data/yolo_dataset/data.yaml
Epochs: 100
Image size: 640
Batch size: 16
Device: 0
Run name: rice_yolo10x
Best weights -> /content/rice_disease_project/models/yolo/yolo10x.pt
Classes: 6 active / 6 total
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rice_disease_project/data/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_

- Tiếp tục từ checkpoint

In [ ]:
!python src/tier1_detection/train_yolo.py --resume

- Evaluate YOLO

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split val

Model: /content/rice_disease_project/models/yolo/yolo9c.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv9c summary (fused): 156 layers, 25,323,874 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3547.5±1130.1 MB/s, size: 248.9 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/val.cache... 227 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 227/227 45.3Mit/s 0.0s
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00043_brown-439_JPG_JPG.rf.WQGk2BMRPnUjIpsujVFr.JPG: 1 duplicate labels removed
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00049_brown-413_JPG_JPG.rf.wddidYmNu5NWOwIenD8Y.JPG: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 745, len(boxes) = 1163. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply e

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split test

Model: /content/rice_disease_project/models/yolo/yolo9c.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv9c summary (fused): 156 layers, 25,323,874 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3056.0±306.9 MB/s, size: 350.5 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/test... 226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 226/226 1.0Kit/s 0.2s
val: /content/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00075_brown-8_JPG_JPG.rf.9NXNUv5qVHKLLEhpvNAk.JPG: 3 duplicate labels removed
val: New cache created: /content/rice_disease_project/data/yolo_dataset/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 873, len(boxes) = 1315. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
  

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split val --model yolo9e

Model: /content/rice_disease_project/models/yolo/yolo9e.pt
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv9e summary (fused): 279 layers, 57,381,026 parameters, 0 gradients, 189.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3574.2±615.3 MB/s, size: 436.7 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/val.cache... 227 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 227/227 43.3Mit/s 0.0s
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00059_brown-413_JPG_JPG.rf.wddidYmNu5NWOwIenD8Y.JPG: 1 duplicate labels removed
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00066_brown-439_JPG_JPG.rf.WQGk2BMRPnUjIpsujVFr.JPG: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 745, len(boxes) = 1163. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply ei

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split test --model yolo9e

Model: /content/rice_disease_project/models/yolo/yolo9e.pt
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv9e summary (fused): 279 layers, 57,381,026 parameters, 0 gradients, 189.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3022.9±543.0 MB/s, size: 384.3 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/test... 226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 226/226 1.1Kit/s 0.2s
val: /content/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00072_brown-8_JPG_JPG.rf.9NXNUv5qVHKLLEhpvNAk.JPG: 3 duplicate labels removed
val: New cache created: /content/rice_disease_project/data/yolo_dataset/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 873, len(boxes) = 1315. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
  

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split val --model yolo10l

Model: /content/rice_disease_project/models/yolo/yolo10l.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10l summary (fused): 174 layers, 24,313,954 parameters, 0 gradients, 120.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3091.0±1067.1 MB/s, size: 258.0 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/val.cache... 227 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 227/227 43.3Mit/s 0.0s
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00043_brown-439_JPG_JPG.rf.WQGk2BMRPnUjIpsujVFr.JPG: 1 duplicate labels removed
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00049_brown-413_JPG_JPG.rf.wddidYmNu5NWOwIenD8Y.JPG: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 745, len(boxes) = 1163. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split test --model yolo10l

Model: /content/rice_disease_project/models/yolo/yolo10l.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10l summary (fused): 174 layers, 24,313,954 parameters, 0 gradients, 120.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3076.3±332.7 MB/s, size: 338.6 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/test... 226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 226/226 1.0Kit/s 0.2s
val: /content/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00075_brown-8_JPG_JPG.rf.9NXNUv5qVHKLLEhpvNAk.JPG: 3 duplicate labels removed
val: New cache created: /content/rice_disease_project/data/yolo_dataset/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 873, len(boxes) = 1315. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split val --model yolo10x

Model: /content/rice_disease_project/models/yolo/yolo10x.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10x summary (fused): 192 layers, 29,402,306 parameters, 0 gradients, 160.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3083.0±985.6 MB/s, size: 409.4 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/val.cache... 227 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 227/227 39.7Mit/s 0.0s
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00043_brown-439_JPG_JPG.rf.WQGk2BMRPnUjIpsujVFr.JPG: 1 duplicate labels removed
val: /content/rice_disease_project/data/yolo_dataset/images/val/brown_spot_00049_brown-413_JPG_JPG.rf.wddidYmNu5NWOwIenD8Y.JPG: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 745, len(boxes) = 1163. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply 

In [ ]:
!python /content/rice_disease_project/src/tier1_detection/evaluate_yolo.py --split test --model yolo10x

Model: /content/rice_disease_project/models/yolo/yolo10x.pt
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10x summary (fused): 192 layers, 29,402,306 parameters, 0 gradients, 160.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2771.2±937.2 MB/s, size: 238.8 KB)
val: Scanning /content/rice_disease_project/data/yolo_dataset/labels/test... 226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 226/226 1.0Kit/s 0.2s
val: /content/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00075_brown-8_JPG_JPG.rf.9NXNUv5qVHKLLEhpvNAk.JPG: 3 duplicate labels removed
val: New cache created: /content/rice_disease_project/data/yolo_dataset/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 873, len(boxes) = 1315. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


V. Tạo crops cho EfficientNet

In [ ]:
!python /content/rice_disease_project/src/utils/crop_for_classifier.py

CROPPING DISEASE REGIONS FROM YOLO LABELS
[train] 10933 crops -> /content/rice_disease_project/data/crops/train
[val] 1165 crops -> /content/rice_disease_project/data/crops/val
[test] 1318 crops -> /content/rice_disease_project/data/crops/test
Crop output directory: /content/rice_disease_project/data/crops
Use for training: python src/tier2_classification/train_efficientnet.py --train_dir /content/rice_disease_project/data/crops/train --val_dir /content/rice_disease_project/data/crops/val


VI. Train EfficientNet-B4

In [ ]:
!python /content/rice_disease_project/src/tier2_classification/train_efficientnet.py \
  --train_dir /content/rice_disease_project/data/crops/train \
  --val_dir   /content/rice_disease_project/data/crops/val

🚀 TẦNG 2: TRAIN EFFICIENTNET-B4 CLASSIFIER
  Device: cuda
  Epochs: 30 (freeze 5 + finetune 25)
  LR: 0.0001
DataLoaders ready: train=10933, val=1165
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100% 74.5M/74.5M [00:00<00:00, 212MB/s]
  Epoch   1/30 | Train 0.9020/0.5155 | Val 1.5480/0.4249 | LR 9.05e-04 | 66.6s
  💾 Best model saved! val_acc=0.4249
  Epoch   2/30 | Train 0.5623/0.6897 | Val 1.2728/0.4850 | LR 6.55e-04 | 65.7s
  💾 Best model saved! val_acc=0.4850
  Epoch   3/30 | Train 0.4818/0.7144 | Val 1.1472/0.5356 | LR 3.45e-04 | 66.0s
  💾 Best model saved! val_acc=0.5356
  Epoch   4/30 | Train 0.4528/0.7346 | Val 1.0886/0.5665 | LR 9.55e-05 | 66.3s
  💾 Best model saved! val_acc=0.5665
  Epoch   5/30 | Train 0.4235/0.7531 | Val 1.1031/0.5648 | LR 0.00e+00 | 65.7s

🔓 Epoch 6: Unfreeze backbone — bắt đầu fine-tune toàn bộ
  Epoch   6/30 | Train 0.2649/0.8225 | Val

* Evaluate EfficientNET-B4

In [ ]:
!python /content/rice_disease_project/src/tier2_classification/evaluate_efficientnet.py \
  --val_dir /content/rice_disease_project/data/crops/val \
  --split val

val set: 1165 images | 6 classes

EVALUATION RESULTS
Accuracy: 0.9468
F1 Macro: 0.9308
F1 Weighted: 0.9478

CLASSIFICATION REPORT:
                       precision    recall  f1-score   support

bacterial_leaf_blight       0.98      0.99      0.98       163
           brown_spot       0.99      0.92      0.96       569
              healthy       0.80      1.00      0.89        40
           leaf_blast       0.89      0.98      0.93       189
           leaf_scald       0.80      0.99      0.88        71
        sheath_blight       0.98      0.91      0.95       133

             accuracy                           0.95      1165
            macro avg       0.91      0.97      0.93      1165
         weighted avg       0.95      0.95      0.95      1165

CONFUSION MATRIX:
                           bacteria  brown_sp   healthy  leaf_bla  leaf_sca  sheath_b
    bacterial_leaf_blight       161         0         0         1         1         0
               brown_spot         3       525 

In [ ]:
!python /content/rice_disease_project/src/tier2_classification/evaluate_efficientnet.py \
  --val_dir /content/rice_disease_project/data/crops/test \
  --split test

test set: 1318 images | 6 classes

EVALUATION RESULTS
Accuracy: 0.9294
F1 Macro: 0.9039
F1 Weighted: 0.9329

CLASSIFICATION REPORT:
                       precision    recall  f1-score   support

bacterial_leaf_blight       0.98      0.99      0.99       187
           brown_spot       0.98      0.88      0.93       642
              healthy       0.53      1.00      0.69        40
           leaf_blast       0.87      0.95      0.91       216
           leaf_scald       0.88      0.99      0.93        90
        sheath_blight       0.99      0.96      0.98       143

             accuracy                           0.93      1318
            macro avg       0.87      0.96      0.90      1318
         weighted avg       0.94      0.93      0.93      1318

CONFUSION MATRIX:
                           bacteria  brown_sp   healthy  leaf_bla  leaf_sca  sheath_b
    bacterial_leaf_blight       186         0         0         0         1         0
               brown_spot         4       568

VII. Chạy Pipeline

- 1 ảnh: dùng template caption

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
--image '/content/rice_disease_project/data/yolo_dataset/images/test/bacterial_leaf_blight_00000_bacterial-265_JPG_JPG.rf.nEQYqWSLP7ibhfKZ5Q8O.JPG' \
--model /content/rice_disease_project/models/yolo/yolo10x.pt

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo10x.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: template
YOLO detections: 1
bbox [658, 0, 932, 1600] -> Bacterial leaf blight (100.0%)
Processing time: 2090.7 ms
Annotated image: /content/rice_disease_project/outputs/visualizations/annotated_bacterial_leaf_blight_00000_bacterial-265_JPG_JPG.rf.nEQYqWSLP7ibhfKZ5Q8O.JPG
Caption file: /content/rice_disease_project/outputs/captions/bacterial_leaf_blight_00000_bacterial-265_JPG_JPG.rf.nEQYqWSLP7ibhfKZ5Q8O_captions.txt


- 1 ảnh: dùng BLIP-2

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
  --image '/content/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00065_brown-281_JPG_JPG.rf.dZKY4ja9Iilav4dlvM0G.JPG' \
  --vlm_mode blip2 \
  --model /content/rice_disease_project/models/yolo/yolo10x.pt

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo10x.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: blip2
Loading BLIP-2 (Salesforce/blip2-opt-2.7b)...
Fetching 2 files: 100% 2/2 [00:27<00:00, 14.00s/it]
Download complete: 100% 10.0G/10.0G [00:28<00:00, 357MB/s]
Loading weights: 100% 1247/1247 [00:04<00:00, 287.28it/s]
generation_config.json: 100% 141/141 [00:00<00:00, 949kB/s]
BLIP-2 loaded
YOLO detections: 1
bbox [772, 1299, 857, 1437] -> Leaf blast (85.7%)
Processing time: 18512.5 ms
Annotated image: /content/rice_disease_project/outputs/visualizations/annotated_brown_spot_00065_brown-281_JPG_JPG.rf.dZKY4ja9Iilav4dlvM0G.JPG
Caption file: /content/rice_disease_project/outputs/captions/brown_spot_00065_brown-281_JPG_JPG.rf.dZKY4ja9Iilav4dlvM0G_captions.txt


- 1 ảnh: dùng LLAVA

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
  --image '/content/rice_disease_project/data/yolo_dataset/images/test/leaf_blast_00153_leaf_blast-433_JPG_JPG.rf.7OPl6klr02UCygvQFJCR.JPG' \
  --vlm_mode llava \
  --model /content/rice_disease_project/models/yolo/yolo10x.pt

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo10x.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: llava
Loading LLaVA (llava-hf/llava-1.5-7b-hf)...
Fetching 3 files: 100% 3/3 [00:36<00:00, 12.16s/it]
Download complete: 100% 14.1G/14.1G [00:36<00:00, 387MB/s]
Loading weights: 100% 686/686 [00:04<00:00, 167.71it/s]
generation_config.json: 100% 141/141 [00:00<00:00, 696kB/s]
LLaVA loaded
YOLO detections: 1
bbox [691, 705, 839, 826] -> Leaf blast (100.0%)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Processing time: 8740.2 ms
Annotated image: /content/rice_disease_project/outputs/visualizations/annotated_leaf_blast_00153_leaf_blast-433_JPG_JPG.rf.7OP

- Cả folder + lưu JSON

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
  --folder '/content/rice_disease_project/data/12-test' \
  --model /content/rice_disease_project/models/yolo/yolo9c.pt \
  --save_json

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo9c.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: template
Processing folder: /content/rice_disease_project/data/12-test (12 images)
[1/12] bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv.JPG
YOLO detections: 2
bbox [845, 653, 930, 1407] -> Bacterial leaf blight (100.0%)
bbox [537, 0, 643, 827] -> Bacterial leaf blight (100.0%)
Processing time: 865.1 ms
Annotated image: /content/rice_disease_project/outputs/visualizations/annotated_bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv.JPG
Caption file: /content/rice_disease_project/outputs/captions/bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv_captions.txt
[2/12] bacterial_leaf_blight_00006_bacterial-251_JPG_JPG.rf.cAOhMRd9swn5vdIqoRRS.JPG
YOLO detections: 9
bbox [628, 76, 663, 32

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
  --folder '/content/rice_disease_project/data/12-test' \
  --vlm_mode blip2 \
  --model /content/rice_disease_project/models/yolo/yolo9c.pt \
  --save_json

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo9c.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: blip2
Loading BLIP-2 (Salesforce/blip2-opt-2.7b)...
Fetching 2 files: 100% 2/2 [00:00<00:00, 3853.29it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 1247/1247 [00:04<00:00, 286.78it/s]
BLIP-2 loaded
Processing folder: /content/rice_disease_project/data/12-test (12 images)
[1/12] bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv.JPG
YOLO detections: 2
bbox [845, 653, 930, 1407] -> Bacterial leaf blight (100.0%)
bbox [537, 0, 643, 827] -> Bacterial leaf blight (100.0%)
Processing time: 36546.2 ms
Annotated image: /content/rice_disease_project/outputs/visualizations/annotated_bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv.JPG
Caption file: /content/rice_disease_project/outputs/captions/b

In [ ]:
!python /content/rice_disease_project/src/pipeline/pipeline.py \
  --folder '/content/rice_disease_project/data/12-test' \
  --vlm_mode llava \
  --model /content/rice_disease_project/models/yolo/yolo9c.pt \
  --save_json

Pipeline device: cuda
Loading YOLO...
YOLO loaded: /content/rice_disease_project/models/yolo/yolo9c.pt
Loading EfficientNet...
EfficientNet loaded: /content/rice_disease_project/models/efficientnet/efficientnet_b4_best.pth
Loading VLM: llava
Loading LLaVA (llava-hf/llava-1.5-7b-hf)...
Fetching 3 files: 100% 3/3 [00:00<00:00, 3950.68it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 686/686 [00:04<00:00, 169.19it/s]
LLaVA loaded
Processing folder: /content/rice_disease_project/data/12-test (12 images)
[1/12] bacterial_leaf_blight_00005_bacterial-252_JPG_JPG.rf.D2zhnTIL5ourSbukIZnv.JPG
YOLO detections: 2
bbox [845, 653, 930, 1407] -> Bacterial leaf blight (100.0%)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
bbox [537, 0, 643, 827] -> Bacterial leaf blight (100.0%)
P

VIII. Backup toàn bộ về Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
cd /content

# Cài pv nếu chưa có
apt-get install -q pv

OUTFILE="rice_disease_project_$(date +%Y%m%d_%H%M).tar.gz"

# Tính tổng size trước để pv hiển thị %
TOTAL=$(du -sb rice_disease_project/ \
  --exclude="rice_disease_project/data/raw_datasets" \
  --exclude="*.cache" \
  --exclude="__pycache__" \
  2>/dev/null | awk '{print $1}')

tar -c \
  --exclude="rice_disease_project/data/raw_datasets" \
  --exclude="*.cache" \
  --exclude="__pycache__" \
  rice_disease_project/ \
  | pv -s "$TOTAL" -p -t -e -r \
  | gzip > "$OUTFILE"

echo "✅ Xong! Kích thước:"
du -sh /content/rice_disease_project_*.tar.gz

Reading package lists...
Building dependency tree...
Reading state information...
Suggested packages:
  doc-base
The following NEW packages will be installed:
  pv
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 44.5 kB of archives.
After this operation, 139 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 pv amd64 1.6.6-1build2 [44.5 kB]
Fetched 44.5 kB in 0s (1,247 kB/s)
Selecting previously unselected package pv.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../pv_1.6.6-1build2_amd64.deb ...
Unpacking pv (1.6.6-1build2) ...
Setting up pv (1.6.6-1build2) ...
Processing triggers for man-db (2.10.2-1) ...
✅ Xong! Kích thước:
1.3G	/content/rice_disease_project_20260628_1643.tar.gz


In [ ]:
# Tạo thư mục đích trên Drive (nếu chưa có)
!mkdir -p "/content/drive/MyDrive/Thesis/rice_disease_backup"

In [ ]:
!cp /content/rice_disease_project_20260628_1643.tar.gz \
    "/content/drive/MyDrive/Thesis/rice_disease_backup"

In [ ]:
!python /Users/whynot.son/Downloads/Yolo/rice_disease_project/src/utils/only_box_visualize_disease.py \
  --image /Users/whynot.son/Downloads/Yolo/rice_disease_project/data/yolo_dataset/images/test/brown_spot_00042_brown-348_JPG_JPG.rf.mvPvixdkq82dkkxBoQBK.JPG \
  --model /Users/whynot.son/Downloads/Yolo/rice_disease_project/models/yolo/yolo10x.pt

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[OK] brown_spot_00042_brown-348_JPG_JPG.rf.mvPvixdkq82dkkxBoQBK.JPG → 10 detections → outputs/visualizations/brown_spot_00042_brown-348_JPG_JPG.rf.mvPvixdkq82dkkxBoQBK_detected.JPG


In [ ]:
!python /Users/whynot.son/Downloads/Yolo/rice_disease_project/src/utils/generate_augmentation_figure.py \
  --image /Users/whynot.son/Downloads/Yolo/rice_disease_project/data/yolo_dataset/images/test/healthy_00111_healthy-331_JPG.rf.9D1GCJqGVNaZbkKNjWPG.JPG \
  --output /Users/whynot.son/Downloads/Yolo/rice_disease_project/outputs/visualizations/augmentation_figure.png

[INFO] Loaded: healthy_00111_healthy-331_JPG.rf.9D1GCJqGVNaZbkKNjWPG.JPG  →  (380, 380, 3)
[OK] Saved → /Users/whynot.son/Downloads/Yolo/rice_disease_project/outputs/visualizations/augmentation_figure_yolo.png
[OK] Saved → /Users/whynot.son/Downloads/Yolo/rice_disease_project/outputs/visualizations/augmentation_figure_efficientnet.png
[DONE] Both figures saved.
